In [1]:
import torch
from torch.utils.cpp_extension import load

differentiable_rapid_raditor = load(
    name="differentiable_rapid_raditor",
    sources=["utils/differentiable_rapid_raditor_kernel.cpp", "utils/differentiable_rapid_raditor_kernel_v3.cu"],
    verbose=True,
)

Using /home/ultraman/.cache/torch_extensions/py310_cu121 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/ultraman/.cache/torch_extensions/py310_cu121/differentiable_rapid_raditor/build.ninja...
Building extension module differentiable_rapid_raditor...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.


Loading extension module differentiable_rapid_raditor...


In [2]:
# Define the custom Autograd Function
class SimulateFunction(torch.autograd.Function):
    
    @staticmethod
    def forward(ctx, sensor_location, source_location, source_p0, source_dx, dt, num_sensors, num_sources, num_times):
        # Call the C++ forward function
        simulate_record = differentiable_rapid_raditor.simulate(
            sensor_location, source_location, source_p0, source_dx, dt, num_sensors, num_sources, num_times)
        
        # Save inputs for backward
        ctx.save_for_backward(sensor_location, source_location, source_p0, source_dx)
        ctx.dt = dt
        ctx.num_sensors = num_sensors
        ctx.num_sources = num_sources
        ctx.num_times = num_times
        
        return simulate_record  # simulate_record：torch.Size([num_sensors * num_times])
    
    @staticmethod
    def backward(ctx, dL_dsimulate_record):
        # dL_dsimulate_record：torch.Size([num_sensors * num_times])
        sensor_location, source_location, source_p0, source_dx = ctx.saved_tensors
        dt = ctx.dt
        num_sensors = ctx.num_sensors
        num_sources = ctx.num_sources
        num_times = ctx.num_times


        # Call the C++ backward function
        grad_source_location, grad_source_p0, grad_source_dx = differentiable_rapid_raditor.simulate_backward(
            sensor_location, source_location, source_p0, source_dx, dL_dsimulate_record.contiguous(), dt, num_sensors, num_sources, num_times
        )
        return None, grad_source_location, grad_source_p0, grad_source_dx, None, None, None, None

# Utility function to use the custom autograd function
def simulate(sensor_location, source_location, source_p0, source_dx, dt, num_sensors, num_sources, num_times):
    return SimulateFunction.apply(sensor_location, source_location, source_p0, source_dx, dt, num_sensors, num_sources, num_times)

In [3]:
from scipy.spatial import Delaunay
import torch
import numpy as np
from torch.autograd import Function
from utils.dataset_loader_full_angle import calculate_detector_location, read_one_dat_into_a_matrix
from utils.loss_utils import l2_loss, ssim
import torch.optim as optim
import trimesh
import random
from utils.sliding_ball_model_coarse_flexible_array_cuda0 import SlidingBallModel  
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
import os

device = torch.device('cuda:0')

def load_ply_vertices_faces(ply_path):
    """读取PLY文件中的顶点和面数据"""
    return trimesh.load(ply_path)

def generate_random_balls(num_balls, ply_path, res, batch_size=10000):
    """在mesh内生成随机球体（优化内存版本）"""
    mesh = load_ply_vertices_faces(ply_path)
    bounds = mesh.bounds
    x_min, x_max = bounds[0,0], bounds[1,0]
    y_min, y_max = bounds[0,1], bounds[1,1]
    z_min, z_max = bounds[0,2], bounds[1,2]
    
    interior_points = []
    with tqdm(total=num_balls, desc="Generating interior points") as pbar:
        while len(interior_points) < num_balls:
            # 批量生成候选点
            candidate_points = np.random.uniform(
                low=[x_min, y_min, z_min],
                high=[x_max, y_max, z_max],
                size=(batch_size, 3)
            ).astype(np.float32)  # 减少内存使用
            
            # 批量检查点是否在mesh内
            inside_mask = mesh.contains(candidate_points)
            interior_points.extend(candidate_points[inside_mask])
            
            # 更新进度条
            pbar.update(min(len(interior_points), num_balls) - pbar.n)
    
    # 随机选择所需数量的点
    selected_indices = np.random.choice(len(interior_points), num_balls, replace=False)
    selected_points = np.array(interior_points)[selected_indices]
    
    # 创建球模型
    balls = []
    for point in tqdm(selected_points, desc="Creating ball models"):
        xyz = torch.tensor(point, requires_grad=False, dtype=torch.float32, device=device)
        pressure_0 = torch.tensor(random.uniform(20, 100), requires_grad=True, dtype=torch.float32, device=device)
        radius = torch.tensor(random.uniform(1*res, 6*res), requires_grad=True, dtype=torch.float32, device=device)
        balls.append(SlidingBallModel(xyz, pressure_0, radius))
    
    return balls


# 迭代函数
def run_iterations(balls, pressure_threshold, radius_max_threshold, radius_min_threshold, mesh):
    new_balls = []
    for ball in balls:
        new_ball = ball.adaptive_density_optimization(pressure_threshold, radius_max_threshold, radius_min_threshold, mesh)
        if new_ball is not None:
            new_balls.append(new_ball)
    balls = [ball for ball in balls if not ball._is_destroyed] + new_balls
    return balls

# 保存点云数据函数
def save_point_cloud(balls, filename):
    points = []
    for ball in balls:
        if not ball._is_destroyed:
            xyz = ball._xyz.detach().cpu().numpy()  # 转为 numpy 数组
            pressure_0 = ball._pressure_0.item()  # 提取标量值
            radius = ball._radius.item()  # 提取标量值
            points.append([xyz[0], xyz[1], xyz[2], pressure_0, radius])

    with open(filename, "w") as ply_file:
        # 写 PLY 文件头
        ply_file.write("ply\n")
        ply_file.write("format ascii 1.0\n")
        ply_file.write(f"element vertex {len(points)}\n")
        ply_file.write("property float x\n")
        ply_file.write("property float y\n")
        ply_file.write("property float z\n")
        ply_file.write("property float pressure_0\n")
        ply_file.write("property float radius\n")
        ply_file.write("end_header\n")
        
        # 写入每个点的信息
        for point in points:
            ply_file.write(f"{point[0]} {point[1]} {point[2]} {point[3]} {point[4]}\n")

# 参数设置

num_times = 4096
Nt = num_times
res = 0.20e-3
Vs = 1500.0
dt = 25e-9  # [s]
num_balls = 200000
folder_name = "iteration_point_cloud_edition48"
os.makedirs(folder_name, exist_ok=True)
# boundaries = (-6.4e-3, 6.4e-3, -6.4e-3, 6.4e-3, -6.4e-3, 6.4e-3)

# 进行初始化，并提取初始化声源的信息
# balls = generate_random_balls(num_balls, boundaries, res)
# 在训练循环开始前（参数设置部分之后）
mesh_path = "data/finger_loose_cover_small_252verticles.ply"
mesh = trimesh.load(mesh_path)  # 确保文件路径正确
print(f"Mesh loaded with {len(mesh.vertices)} vertices")

# 检查bounds是否有效
bounds = mesh.bounds
print(f"Mesh bounds: {bounds}")
balls = generate_random_balls(num_balls, mesh_path, res)
source_num = num_balls

pressure_threshold = 15.0
radius_max_threshold = 3 * res
radius_min_threshold = 0.5 * res


Mesh loaded with 106 vertices
Mesh bounds: [[ 0.012603   -0.004       0.013264  ]
 [ 0.03522741  0.104337    0.08300044]]


Creating ball models:   0%|          | 0/200000 [00:00<?, ?it/s]/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._xyz = Parameter(torch.tensor(xyz, dtype=torch.float32) if xyz is not None else torch.empty(0))
/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._pressure_0 = Parameter(torch.tensor(pressure_0, dtype=torch.float32) if pressure_0 is not None else torch.empty(0))
/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:10: UserWarning: To 

In [4]:
# 读取sensor_data_matrix.txt文件
sensor_data_matrix = np.loadtxt('data/sensor_signal_2006elements_4096.txt', delimiter='\t')
real_signal = sensor_data_matrix#[0:1009:2, :]
sensor_location = np.loadtxt('data/sensor_location_2006elements.txt', delimiter='\t')
sensor_location = sensor_location#[0:1009:2, :]
sensor_num = sensor_location.shape[0]

print(f"GT:{real_signal.shape}")
print(f"location:{sensor_location.shape}")

GT:(2006, 4096)
location:(2006, 3)


In [ ]:

# 将数据转为PyTorch张量，并移动到GPU
sensor_location = torch.tensor(sensor_location,dtype=torch.float, device=device)
real_signal = torch.tensor(real_signal,dtype=torch.float)
real_signal_flat = real_signal.flatten()
real_signal_flat = real_signal_flat.to(device)

# 将所有参数按照不同学习率进行分组
def get_optim_params(balls):
    params_pressure = [ball._pressure_0 for ball in balls]
    params_radius = [ball._radius for ball in balls]
    return [
        {'params': params_pressure, 'lr': 0.05},
        {'params': params_radius, 'lr': 0.00004},
    ]

# 设置学习率调度器
# lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)

num_iterations = 1000

# ========== 修正的预过滤阶段 ==========
print("开始预过滤...")
filter_start_time = time.time()
# 1. 保存原始pressure_0的数值（深拷贝）
original_pressure_values = [ball._pressure_0.data.clone() for ball in balls]
# 2. 将所有pressure_0设为0（保持Parameter属性）
with torch.no_grad():
    for ball in balls:
        ball._pressure_0.data.zero_()
# 3. 创建临时优化器
temp_optimizer = optim.Adam([{'params': [ball._pressure_0 for ball in balls], 'lr': 1.0}])
temp_optimizer.zero_grad()
# 4. 前向计算
source_location = torch.stack([ball._xyz for ball in balls]).flatten()
source_p0 = torch.stack([ball._pressure_0 for ball in balls]).flatten()
radius_0 = torch.stack([ball._radius for ball in balls]).flatten()

temporary_downsample_factor = 16
temporary_ds_num_times = num_times//temporary_downsample_factor
temporary_ds_dt = dt*temporary_downsample_factor
temporary_ds_real_signal = real_signal[:, ::temporary_downsample_factor,]
temporary_ds_real_signal_flat = temporary_ds_real_signal.flatten()
temporary_ds_real_signal_flat = temporary_ds_real_signal_flat.to(device)

simulate_record = simulate(sensor_location, source_location, source_p0, radius_0, temporary_ds_dt, sensor_num, source_num, temporary_ds_num_times)  
loss = l2_loss(simulate_record, temporary_ds_real_signal_flat)
# 5. 反向传播
loss.backward()
# 6. 过滤与恢复
preserved_balls = []
destroyed_count = 0

# 创建保存梯度值的球列表
gradient_balls = []
for i, ball in enumerate(balls):  # 修改这里，添加enumerate获取索引i
    # 复制球的基本属性，用梯度值代替pressure_0
    gradient_ball = type(ball)(xyz=ball._xyz.detach().clone(),
                              pressure_0=ball._pressure_0.grad.clone() if ball._pressure_0.grad is not None else torch.zeros_like(ball._pressure_0),
                              radius=ball._radius.detach().clone())
    gradient_balls.append(gradient_ball)
    
    # 原始过滤逻辑
    grad = ball._pressure_0.grad
    if grad is not None and grad.item() <= 0:
        # 恢复原始随机初始化的压强值
        with torch.no_grad():
            ball._pressure_0.data.copy_(original_pressure_values[i])  
        preserved_balls.append(ball)
    else:
        destroyed_count += 1

balls = preserved_balls
source_num = len(balls)

# 保存梯度点云
filename = "ball_0_pressure_gradients_huge_num.ply"
full_path = os.path.join(folder_name, filename)
save_point_cloud(gradient_balls, full_path)
print(f"Pressure gradients saved to {filename}")

# 验证恢复结果
sample_pressures = [ball._pressure_0.item() for ball in balls[:5]]
print(f"预过滤完成，销毁球数量: {destroyed_count}, 保留球数量: {source_num}")
filter_end_time = time.time()  # 记录结束时间
filtering_time = filter_end_time - filter_start_time  # 计算时间差
print(f"过滤时间: {filtering_time}")
print("示例恢复的压强值:", sample_pressures)
# 保存预过滤结果
filename = os.path.join(folder_name, "ball_0_after_filter.ply")
save_point_cloud(balls, filename)
print(f"点云已保存到 {filename}")

# ---------- 以下是正式迭代过程 ----------
optimizer = optim.Adam(get_optim_params(balls), betas=(0.9, 0.999))


# 开始迭代训练
for iter in range(num_iterations):
    if (iter + 1) <= 80 :
        downsample_factor = 16
        ds_num_times = num_times//downsample_factor
        ds_dt = dt*downsample_factor
        ds_real_signal = real_signal[:, ::downsample_factor,]

    if (iter + 1) <= 100 and (iter + 1) > 80:
        downsample_factor = 4
        ds_num_times = num_times//downsample_factor
        ds_dt = dt*downsample_factor
        ds_real_signal = real_signal[:, ::downsample_factor,]

    if (iter + 1) > 100:
        downsample_factor = 1
        ds_num_times = num_times//downsample_factor
        ds_dt = dt*downsample_factor
        ds_real_signal = real_signal[:, ::downsample_factor,]

    ds_real_signal_flat = ds_real_signal.flatten()
    ds_real_signal_flat = ds_real_signal_flat.to(device)
    start_time = time.time()  # 记录开始时间
    optimizer.zero_grad()

    # 使用当前球的参数传递给simulate函数
    source_location = torch.stack([ball._xyz for ball in balls]).flatten()
    source_p0 = torch.stack([ball._pressure_0 for ball in balls]).flatten()
    radius_0 = torch.stack([ball._radius for ball in balls]).flatten()
    
    simulate_record = simulate(sensor_location, source_location, source_p0, radius_0, ds_dt, sensor_num, source_num, ds_num_times)  
    loss = l2_loss(simulate_record, ds_real_signal_flat)
    loss.backward()
    optimizer.step()
    # lr_scheduler.step()
    
    # print(f"迭代 {iter + 1}，损失: {loss.item()}")
    end_time = time.time()  # 记录结束时间
    iteration_time = end_time - start_time  # 计算时间差
    print(f"迭代 {iter + 1}，损失: {loss.item()}，时间: {iteration_time}")
    # 更新每个球的参数，以确保它们在相应的实例中
    for ball in balls:
        ball._xyz.data = ball._xyz.data.requires_grad_(False)
        ball._pressure_0.data = ball._pressure_0.data.requires_grad_(True)
        ball._radius.data = ball._radius.data.requires_grad_(True)

    # 每10次迭代前保存点云数据到文件
    if (iter + 1) % 10 == 0:
        filename = f"ball_{iter + 1}_before.ply"
        full_path = os.path.join(folder_name, filename)
        save_point_cloud(balls, full_path)
        print(f"Point cloud saved to {filename}")

    # 每10次迭代进行自适应密度优化
    if (iter + 1) % 10 == 0:
        with torch.no_grad():
            balls = run_iterations(
                balls,  
                pressure_threshold,
                radius_max_threshold,
                radius_min_threshold,
                mesh
            )
        source_num = len(balls)
        print(f"Iter {iter+1}: {source_num} balls remaining")
        # 更新优化器中的参数
        optimizer = optim.Adam(get_optim_params(balls), betas=(0.9, 0.999))
        
    # 每10次迭代后保存点云数据到文件
    if (iter + 1) % 10 == 0:
        filename = f"ball_{iter + 1}_after.ply"
        full_path = os.path.join(folder_name, filename)
        save_point_cloud(balls, full_path)
        print(f"Point cloud saved to {filename}")

开始预过滤...


/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._xyz = Parameter(torch.tensor(xyz, dtype=torch.float32) if xyz is not None else torch.empty(0))
/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self._pressure_0 = Parameter(torch.tensor(pressure_0, dtype=torch.float32) if pressure_0 is not None else torch.empty(0))
/home/ultraman/my_file/SlingBAG++_finger/utils/sliding_ball_model_coarse_flexible_array_cuda0.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTen

Pressure gradients saved to ball_0_pressure_gradients_huge_num.ply
预过滤完成，销毁球数量: 93076, 保留球数量: 106924
过滤时间: 52.649789333343506
示例恢复的压强值: [69.62897491455078, 91.13473510742188, 45.343528747558594, 98.59163665771484, 37.904476165771484]
点云已保存到 iteration_point_cloud_edition48/ball_0_after_filter.ply
迭代 1，损失: 6374.8984375，时间: 14.252373695373535
迭代 2，损失: 5334.1845703125，时间: 10.77821683883667
迭代 3，损失: 4448.087890625，时间: 9.424611568450928
迭代 4，损失: 3712.6044921875，时间: 9.489339828491211
迭代 5，损失: 3103.9345703125，时间: 9.430578470230103
迭代 6，损失: 2602.499267578125，时间: 9.442055702209473
迭代 7，损失: 2192.203857421875，时间: 10.678935527801514
迭代 8，损失: 1857.155517578125，时间: 9.397028684616089
迭代 9，损失: 1583.08251953125，时间: 9.39600682258606
迭代 10，损失: 1360.4688720703125，时间: 9.406046628952026
Point cloud saved to ball_10_before.ply
Iter 10: 142704 balls remaining
Point cloud saved to ball_10_after.ply
迭代 11，损失: 516.2542724609375，时间: 20.274896383285522
迭代 12，损失: 476.50299072265625，时间: 12.758021831512451
迭代 13，损失: 4